# 1 · Outcomes  `[EVAL]`

Full-conversation eval outcomes (the held-out measure, never the training reward): the all-metric trajectory grid, a per-metric learning-curve catalog (`trajectories/`), the vs-base effect forest, per-model bars, and the scorecard. Exports → `results/<VIEW>/figures|tables/1_outcomes/`. Deeper splits: heterogeneity → `2`, behaviour/mechanism → `3`, stats tables → `6`.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath("."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)
import eda_analysis
from eda_analysis import stats, figures, plots

# ╔═══ VIEW — the one knob ════════════════════════════════════════════════════════╗
# "all" = every arm | "L0" = K=0 arms (PTO_LA0/GRPO_LA0) | "L5" = K=5 arms (thin, paused).
# Sets BOTH the arm filter AND the results root -> results/<VIEW>/figures|tables/<group>/.
# Edit the default for interactive use; render_views.py overrides it via the EDA_VIEW env var.
VIEW = os.environ.get("EDA_VIEW", "L0")

# Secondary knobs — defaults are fine. ks/methods left default so VIEW drives arm selection
# (set ks=[...] only to override the view's K filter). All fields: eda_analysis/config.py.
cfg = eda_analysis.EdaConfig(
    view=VIEW,                             # arm filter + results/<VIEW>/ root
    export_group="1_outcomes",             # topic family = this notebook's number · figs=PNG, tables=md+xlsx
    selection="all",                       # "all" | "best" (best iter per arm by own oracle)
    focus_arms=None, focus_metric="Q1Q2",  # default arms/metric for single-metric figures
)
S = eda_analysis.notebook_setup(cfg)
FOCUS = cfg.focus_arms or sorted(S.SCORES.arm.unique())   # arms shown in the trajectory figures

## 1 · Outcome trajectories — all metrics  `[EVAL]`
Per-metric mean ± 95% CI across iterations, arms overlaid — the one-glance overview. Global-eval (halo) rubrics beside the orthogonal axes (PCT, MICI ↓); caveat under the grid.

In [ ]:
fig = plots.trajectory_grid(S.SCORES, palette=S.PALETTE, arms=cfg.focus_arms)
eda_analysis.save_fig(fig, "trajectories_all_metrics", caption="Full-conv eval: per-metric mean +/- 95% CI across iterations, arms overlaid (all 9 metrics incl. the orthogonal axes)."); plt.show()

## 2 · Per-metric learning curves  `[EVAL]`
One full-size curve per metric → `figures/1_outcomes/trajectories/`. Peaks that precede the final iteration are auto-flagged ("peak → regresses" — GRPO's iter-8 peak). The oracle-noise band draws only on Q1+Q2 (the ~0.10 reproducibility figure was measured on that rubric).

In [ ]:
for m in S.METRICS:
    fig = plots.single_metric_trajectory(S.SCORES, m, palette=S.PALETTE, arms=cfg.focus_arms,
                                         oracle_noise=(S.ORACLE_NOISE if m == "Q1Q2" else None),
                                         mark_peaks=True)
    eda_analysis.save_fig(
        fig, f"trajectory_{m}", group="1_outcomes/trajectories",
        caption=f"{eda_analysis.display_label(m)} across iterations per arm (mean +/- 95% CI, N=96); "
                "dotted vline flags a peak that precedes the final iteration (regression)."
                + (" Grey band = oracle reproducibility (~0.10) around base." if m == "Q1Q2" else ""))
    plt.show()

## 3 · Did it work? — effect vs base  `[EVAL]`
The vs-base effect per arm × metric as a forest plot (all 9 metrics; MI-Inconsistency ↓ rows are valence-inverted — red = moved the wrong way). Full stats tables in `6_Stats`.

In [ ]:
MR = stats.filter_thin_arms(stats.main_results_table(S.SCORES, target="final"), S.SCORES)
fig = plots.effect_forest(MR)
eda_analysis.save_fig(fig, "effect_vs_base_forest", caption="Improvement vs base (Δ + 95% CI) per arm x rubric, full-conv eval; dot color = effect-size label, dz annotated."); plt.show()

## 4 · Appendix — per-model outcome bars  `[EVAL]`
The exhaustive per-model bar view (every iteration, arm-bases pooled into one descriptive `Base`, dotted base line).

In [ ]:
ALL_D = eda_analysis.collapse_base(eda_analysis.all_models(S.SCORES))
fig = plots.outcomes_by_model(ALL_D, palette=figures.arm_palette(sorted(ALL_D.arm.unique())), order=figures.model_order(ALL_D))
eda_analysis.save_fig(fig, "outcomes_by_model", caption="All models x metrics; mean +/- 95% CI over 96 personas (arm-bases pooled into Base; dotted line = base)."); plt.show()

## 5 · Scorecard — global-eval rubrics vs orthogonal axes  `[EVAL]`
Each arm's best-iteration score per metric: the global-eval (halo) rubrics beside `PCT`, `MICI ↓`, and the derived `R:Q`/`%CR`/`%MICO`. **Read:** the halo cluster can rise while technique / patient-outcome lag — the multi-skill story the global-eval rubrics hide.

In [ ]:
LB = plots.leaderboard_scorecard(S.SCORES, selection="best")
display(LB)
eda_analysis.save_table(LB, "leaderboard_scorecard", caption="Best iteration per arm: global-eval (halo) rubrics beside the orthogonal axes (MICI lower-is-better, flagged ↓).")

## 6 · Artifact index
Refresh the per-view `results/<VIEW>/INDEX.md` (every notebook ends with this — whichever ran last completes the map).

In [ ]:
print("index ->", eda_analysis.build_index())